In [ ]:
#!/usr/bin/env python3
import subprocess
import shutil
from pathlib import Path
import sys

# ─── Paths ───────────────────────────────────────────────────────────────────────
DICOM_ROOT = Path(r"D:\Abdomen_CT_Bone_Mets")
NIFTI_ROOT = Path(r"D:\Abdomen_CT_Bone_Mets_Nifti")
NIFTI_ROOT.mkdir(parents=True, exist_ok=True)

# ─── Check for dcm2niix ──────────────────────────────────────────────────────────
if shutil.which("dcm2niix") is None:
    sys.exit(
        "ERROR: dcm2niix not found. Install via:\n"
        "  conda install -c conda-forge dcm2niix\n"
        "or download from https://github.com/rordenlab/dcm2niix"
    )

# ─── Convert each study ───────────────────────────────────────────────────────────
for study_dir in DICOM_ROOT.iterdir():
    if not study_dir.is_dir():
        continue

    out_folder = NIFTI_ROOT / study_dir.name
    out_folder.mkdir(parents=True, exist_ok=True)

    cmd = [
        "dcm2niix",
        "-z", "y",                   # gzip
        "-r", "y",                   # reorient to canonicalRAS
        "-v", "y",                   # verbose logging
        "-f", study_dir.name,        # output filename
        "-o", str(out_folder),       # output folder
        str(study_dir)               # input DICOM folder
    ]

    print(f"→ Converting {study_dir.name} …")
    try:
        result = subprocess.run(
            cmd,
            check=True,
            capture_output=True,
            text=True
        )
        print(result.stdout)   # dcm2niix info
        print(f"✅  Success: {out_folder / (study_dir.name + '.nii.gz')}\n")

    except subprocess.CalledProcessError as e:
        print(f"❌  Failed on {study_dir.name} (exit {e.returncode}):")
        print("----- STDOUT -----")
        print(e.stdout.strip())
        print("----- STDERR -----")
        print(e.stderr.strip())
        print("\nContinuing to next folder...\n")

In [ ]:
import numpy as np
import nibabel as nib
from pathlib import Path

studies = ["BMAB3_00000240", "BMAB3_00000241"]
root    = Path(r"D:\Abdomen_CT_Bone_Mets_Nifti")

for study in studies:
    nifti_path = root / study / f"{study}.nii.gz"
    img        = nib.load(str(nifti_path))
    aff        = img.affine

    # Raw Z-axis vector & normalized unit
    z_vec  = aff[:3, 2]
    unit_z = z_vec / np.linalg.norm(z_vec)

    # The direction cosine (with scaling) along Z is aff[2,2]
    z_cosine = aff[2, 2]
    z_sign   = np.sign(z_cosine)

    print(f"\n=== {study} ===")
    print(f"Z-cosine (affine[2,2]): {z_cosine:.2f}")
    print(f"Signed Z-cosine:         {z_sign:+.2f}")
    print(f"Raw Z-vector:            {z_vec}")
    print(f"Unit Z-vector:           {unit_z}")
    print("-" * 60)